# All-atom PhantomWalk: fast initialization of dense polymer melts

This tutorial builds a dense all-atom polyethylene melt at its target density in well under a minute on a CPU, using the all-atom PhantomWalk workflow:

1. **Build chains** with `flowermd.base.Polymer` (here the `PolyEthylene` preset).
2. **Place them at the target density** with `AllAtomRandomWalk` (or `AllAtomLattice`). Chains overlap; that is intended.
3. **Build the `AllAtomDPD` force field**: UFF bonded terms scaled up, plus a soft DPD pair force that lets overlapping atoms pass through each other.
4. **Relax with DPD until the energies are stationary, then clean up with FIRE** in `AllAtomPhantomWalk.run_initialization`.
5. **Hand off** the coordinates to a standard force field for minimization.

The result is *minimizer-ready coordinates at the right density*, not an equilibrated melt. Units inside the workflow are Angstrom, kcal/mol and amu; mBuild objects stay in nm.

Requires the `all-atom` extra (`rdkit`; `openff-toolkit` and `openff-interchange` only for `bonded="openff"`).

In [ ]:
import warnings

import unyt as u

from flowermd.library import (
    AllAtomDPD,
    AllAtomPhantomWalk,
    AllAtomRandomWalk,
    PolyEthylene,
)

warnings.filterwarnings("ignore")

## 1. Chains

Ten polyethylene chains of 20 repeat units (about 1,200 atoms). `PolyEthylene` builds each chain with mBuild's polymer recipe, so every repeat unit is a child of the chain compound; the placement step below turns the bonds between those repeats.

In [ ]:
chains = PolyEthylene(lengths=20, num_mols=10)
print("atoms:", chains.n_particles)

## 2. Placement at the target density

`AllAtomRandomWalk` sizes a cubic box for 0.85 g/cm³, gives each chain a random orientation and position, and turns every bond between two repeat units to a random torsion, so the chains start as overlapping random coils. Only those torsions change: bond lengths, bond angles and stereocenters stay as built. `AllAtomLattice` is the alternative: whole chains in their built conformation, one per site of a grid, each with a random rotation.

In [ ]:
system = AllAtomRandomWalk(
    molecules=chains, density=0.85 * u.g / u.cm**3, seed=1
)
print("box (nm):", system.target_box.round(3))

## 3. The all-atom DPD force field

Defaults are the PhantomWalk protocol: UFF bonds, angles and torsions scaled by 30, DPD repulsion `A=1250` and friction `gamma=200` weighted per pair by the UFF Lennard-Jones well depths, `kT=1` kcal/mol, cutoff 3.5 Å. The force field also builds the matching HOOMD frame (`ff.frame`), because interaction types are named by their coefficients rather than by element pairs. Stereocenters, if any, are recorded and protected automatically; polyethylene has none.

In [ ]:
ff = AllAtomDPD(system.system)
print("forces:", list(ff.forces_by_role))
print(
    "particle types:",
    len(ff.frame.particles.types),
    " bond types:",
    len(ff.frame.bonds.types),
)
print("stereocenters protected:", ff.stereo_centers)
print("parameterization time (s):", round(ff.timings["parameterization"], 2))

## 4. DPD until stationary, then FIRE

`run_initialization` runs DPD in chunks of 500 steps after a 4,000-step minimum and stops when the per-particle energy of every force changes by less than 2 % for two consecutive chunks (cap 40,000 steps), then runs 100 FIRE steps with conservative DPD repulsion. If the cap is hit the coordinates are still returned with `dpd_converged=False` and a warning; pass `require_convergence=True` to make that an error instead. Everything about the run lands in the returned record.

In [ ]:
sim = AllAtomPhantomWalk.from_system(
    system, forcefield=ff, dt=0.001, gsd_write_freq=1e5, log_write_freq=1e5
)
record = sim.run_initialization()
print("DPD steps:", record["dpd_steps"], " converged:", record["dpd_converged"])
print("FIRE steps:", record["fire_steps"])
print(
    "wall time (s):", {k: round(v, 1) for k, v in record["timings_s"].items()}
)

The energy history behind the stopping decision is in the record as well (one row per chunk, one column per force, per particle):

In [ ]:
import numpy as np

history = np.array(record["dpd_energy_history"])
print(record["dpd_monitored_forces"])
print(history[-3:].round(4))

## 5. Hand-off

`to_compound()` writes the final, unwrapped coordinates (nm) back into the mBuild compound, ready for a force-field parameterizer such as OpenFF Interchange or foyer. `write_record` keeps the run record next to the structure. `sim.save_restart_gsd()` writes the HOOMD state if you want to continue in HOOMD instead.

In [ ]:
compound = sim.to_compound()
compound.save("pe_melt_initialized.mol2", overwrite=True)
sim.write_record("pe_melt_record.json")
print(compound)

## Knobs for experiments

Everything the PhantomWalk study varied is an argument:

- `AllAtomDPD(bonded="openff")` uses Sage 2.3.0 bonded terms instead of UFF (needs OpenFF).
- `A`, `gamma`, `kT`, `r_cut`, `bonded_scale`, `epsilon_weighting=False`.
- `include_bonds`, `include_angles`, `include_dihedrals`, `include_impropers` for ablations.
- `AllAtomRandomWalk` vs `AllAtomLattice`, and their seeds.
- `run_initialization(dpd_min_steps, dpd_chunk, dpd_max_steps, energy_tol, consecutive, stop=callable, fire_steps, fire_dt, fire_kwargs)`.
- `protect_stereochemistry`, `stereo_k` for chemistries with stereocenters.